# Track 01 — EEG-to-Image · pipeline proof

**What this notebook proves:** that we can pull the data, pull a model, train it, evaluate it
with the competition's metric, and package a submission — end to end, on Colab, without surprises.

**What it does not do:** chase performance. Every model here is a stock baseline.

---

### The task

Given one EEG epoch recorded while a participant viewed a natural image, predict a **1536-d
embedding** in frozen `facebook/dinov2-giant` space, then rank held-out candidate images by
similarity. Training and test images do not overlap — the shift is *cross-stimulus*.

| | |
|---|---|
| Ranking metric | Top-5 retrieval accuracy against the full held-out gallery |
| NeuralBench key | `test/full_retrieval/top5_acc_subject-agg` |
| Target space | `facebook/dinov2-giant`, relative depth 0.6667, mean token pooling, imsize 518 |
| Hidden cohort | 11 participants, 32-ch Emotiv @ 256 Hz (Alljoined) |

⚠️ `val/batch_top5_acc` ranks **within a batch** only. It reads far higher and is not comparable
to the competition score. Never quote it.

## Two things to know before running

**1. Runs are subsetted by default.** `FRACTION` below controls how much data is used. At 2% this
notebook is a pipeline proof — it answers *does everything wire together*, not *is the model good*.
Set `FRACTION = 1.0` for a real run. The subsetting is deterministic and nested, so a 2% run and a
later 10% run share cached work.

**2. Track 1 subsetting has a trap, and the notebook guards against it.** The score is retrieval
against the held-out gallery, so chance is `k / gallery_size`. Shrinking the gallery inflates the
score for reasons unrelated to the model:

| gallery | chance Top-5 |
|---|---|
| 16,740 | 0.03% |
| 200 | 2.50% |
| 10 | **50.00%** |

So we subset the **training split only** — fewer subjects, fewer trials — and leave the test gallery
whole. `assert_gallery_intact` fails the run if a subset ever reaches the test split.

Because of this, **every score in this notebook is reported with its gallery size attached.** A Top-5
number without one is meaningless.

> **Status: scaffold, not yet executed.** Cells marked `# VERIFY` use API surface taken from the
> docs but not run end to end. Expect to fix a few before this goes green.

---

# Stage 0 — smoke test on a couple of recordings

**Run this first.** CPU only, no GPU, no Drive, ~20 MB. It works on a laptop.

Alljoined-1.6M is on EEGDash as **`nm000134`**: 20 subjects, 1525 recordings, 32 ch @ 256 Hz,
8.8 GB total — so roughly **6 MB per recording**. Pulling three is a couple of data points, and it is
the same corpus and hardware as the hidden evaluation cohort.

This matters because the subsetting in Stage 1 happens *after* download. Selecting exact records is
the only way to touch a large corpus cheaply.

Stage 0's real job is **discovery**: print what the data actually looks like, so the remaining
`# VERIFY` cells can be written against reality instead of guessed from docs.

⚠️ Alljoined-1.6M is **CC-BY-NC-ND-4.0** — non-commercial *and* no-derivatives. Stricter than the
other corpora here. Worth checking before anything derived from it gets published.

In [ ]:
%pip install -q 'eegdash>=0.9.1'

import os, json
from pathlib import Path

CACHE = Path(os.environ.setdefault('EEGDASH_CACHE_DIR', str(Path.home() / '.eegdash_cache')))
CACHE.mkdir(parents=True, exist_ok=True)
print('cache:', CACHE)

### 0.1 — What is in the dataset

Query first, download nothing. This tells us the subject/session/task vocabulary to select on.

In [ ]:
from eegdash import EEGDash

DATASET = 'nm000134'   # Alljoined-1.6M

records = EEGDash().find({'dataset': DATASET})
print('records:', len(records))

# Keep the converted BIDS files, not the original source format.
records = [r for r in records if r['bids_relpath'].startswith('sub-')]
print('bids records:', len(records))
print('\nfields on a record:')
print(json.dumps({k: str(v)[:60] for k, v in records[0].items()}, indent=2))

In [ ]:
# Vocabulary we can select on.
from collections import Counter
for field in ('subject', 'session', 'task', 'run'):
    vals = Counter(r.get(field) for r in records)
    print(f'{field:>8}: {len(vals)} unique -> {sorted(str(v) for v in vals)[:8]}')

### 0.2 — Pull three recordings

One subject, so the download stays small and we can still see within-subject structure.

In [ ]:
from miniload import select_under_budget, estimate_mb, host_limits, epochs_mb, targets_mb

print('host:', {k: (v if isinstance(v, str) else round(v, 1)) for k, v in host_limits().items()})

# Budget first, download second. Spread across subjects so the batch exercises
# variation, not just one person's recordings.
sel = select_under_budget(records, budget_mb=60, spread_by='subject')
picked = sel.records

print(f'\n{len(picked)} records, ~{sel.est_mb:.0f} MB est '
      f'({len({r["subject"] for r in picked})} subjects, {sel.skipped_over_budget} skipped)')
for r in picked:
    print(f"  {estimate_mb(r):5.1f} MB  {r['bids_relpath']}")

**Memory, measured rather than guessed.** One 317 s recording at 32 ch / 256 Hz is 10.4 MB as
float32. Epoched into 1 s windows it becomes ~42 MB per recording, and the full set of 16,740
DINOv2-giant targets is 103 MB (which matches the ~100 MB the NeuralBench docs quote for
THINGS-EEG2, a useful cross-check).

None of that is near a Colab limit. The whole corpus held in RAM would be ~15.6 GB, which is why we
do not hold it. The one item of consequence is the DINOv2-giant checkpoint at ~4.5 GB; on a GPU
runtime it lands in VRAM, on a CPU runtime it competes with everything else.

In [ ]:
from eegdash import EEGDashDataset

ds = EEGDashDataset(records=picked, cache_dir=CACHE)
print(ds.description.to_string(index=False))

### 0.3 — Inspect one recording

Signal shape, sampling rate, channel names, and — the part I most need — the **event schema**.
Alljoined's stimulus-identity column will differ from THINGS-EEG2's `tot_img_number`, and the
target extraction cannot be written until we know what it is called.

In [ ]:
raw = ds.datasets[0].raw          # VERIFY: attribute name for the underlying MNE Raw
print('sfreq   :', raw.info['sfreq'])
print('n_chans :', len(raw.ch_names))
print('channels:', raw.ch_names)
print('duration:', raw.n_times / raw.info['sfreq'], 's')
print('has montage positions:', raw.get_montage() is not None)

In [ ]:
import pandas as pd

# The events sidecar is what maps an EEG epoch to the image that was shown.
ev = raw.annotations.to_data_frame() if len(raw.annotations) else None
print('annotations:', len(raw.annotations))
if ev is not None:
    print(ev.head(10).to_string())
    print('\ncolumns:', list(ev.columns))

# VERIFY: locate the stimulus-identity column and the stimuli.tsv mapping for Alljoined.
# THINGS-EEG2 used tot_img_number + stimuli.tsv; Alljoined's schema is unknown here.

### 0.4 — Paste this back

One cell, everything needed to finish the stubbed cells in Stage 1.

In [ ]:
import platform, sys

report = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'n_records_total': len(records),
    'select_fields': {f: sorted({str(r.get(f)) for r in records})[:10]
                      for f in ('subject', 'session', 'task', 'run')},
    'record_keys': list(records[0].keys()),
    'sfreq': raw.info['sfreq'],
    'ch_names': raw.ch_names,
    'n_annotations': len(raw.annotations),
    'event_columns': list(ev.columns) if ev is not None else None,
    'montage_present': raw.get_montage() is not None,
}
print(json.dumps(report, indent=2, default=str))

---

## 0.5 — Fixes from the first run

Three findings from the Stage 0 report, in order of importance.

**Montage is absent.** `montage_present: False`. REVE encodes 3-D electrode coordinates, so with no
montage every channel becomes `INVALID_VALUE` and the model's main advantage silently disappears.
The 32 names are standard 10-10 and all resolve against `standard_1005` (verified offline, 32/32,
no NaNs), so setting it explicitly fixes this. Same pattern the stock Track 3 config uses for
Sleep-EDF.

**Session `02old` exists** alongside `01`-`04`. A superseded duplicate. Exclude it.

**Events are not in `raw.annotations`.** 61 annotations with only `onset/duration/description`, but
this corpus averages ~1000 image presentations per recording. The stimulus identity lives in the BIDS
`events.tsv` sidecar.

In [ ]:
# Record filtering, and a size budget computed before anything downloads.
EXCLUDE_SESSIONS = {'02old'}

clean = [
    r for r in records
    if not r.get('_has_missing_files')
    and r.get('session') not in EXCLUDE_SESSIONS
]
print(f'{len(records)} records -> {len(clean)} after filtering')

def est_mb(r):
    # float32 per sample per channel; a rough but useful pre-download budget.
    n = r.get('ntimes') or 0
    c = r.get('nchans') or 0
    return n * c * 4 / 1e6

tot = sum(est_mb(r) for r in clean)
print(f'full filtered set: ~{tot/1000:.1f} GB across {len(clean)} records')
print(f'median record: ~{sorted(est_mb(r) for r in clean)[len(clean)//2]:.1f} MB')

In [ ]:
# Set the montage explicitly and confirm every channel resolves to real coordinates.
import numpy as np, mne

MONTAGE = 'standard_1020'   # 32/32 resolve; standard_1005 is deprecated from MNE 1.14

raw.set_montage(mne.channels.make_standard_montage(MONTAGE), match_case=False)

pos = raw.get_montage().get_positions()['ch_pos']
P = np.array([pos[c] for c in raw.ch_names])
assert not np.isnan(P).any(), 'some channels still have no coordinates'
print(f'{len(P)}/{len(raw.ch_names)} channels positioned, no NaNs')
print(f'posterior (y<0): {(P[:,1]<0).sum()}  anterior (y>0): {(P[:,1]>0).sum()}')
# Heavily posterior - a visual-task montage, unlike the frontal/temporal-heavy
# clinical layouts most EEG foundation models pretrain on.

### Annotations decoded

`description` is a triple: `<condition>,<id>,<value>`. Observed conditions are `behav` (ids 2, 3) and
`oddball` (id 16740).

These are **not** the stimulus stream — 61 annotations over 317 s is one every 5.2 s, far too sparse
for a 1.6M-trial corpus. They read as behavioural responses and catch trials.

`16740` is the THINGS catalogue size, so the oddball id is a sentinel rather than a real image index.
The corpus is part of the THINGS initiative (arXiv:2508.18571), recorded on a 32-channel
consumer-grade *wet* system - which is why the channel names are standard 10-10.

⚠️ Use `raw.annotations.onset` for seconds. `to_data_frame()` returns absolute datetimes here because
`meas_date` is set, which is useless for epoching.

In [ ]:
import pandas as pd

ann = pd.DataFrame({
    'onset_s': raw.annotations.onset,          # seconds, not datetimes
    'duration': raw.annotations.duration,
    'description': raw.annotations.description,
})
# Four fields: condition, stimulus id, block, trial index. The TSV writes the last
# two with a comma between them, which reads as a decimal comma until you count.
parts = ann['description'].str.split(',', expand=True)
ann[['condition', 'stim_id', 'block', 'trial']] = parts.iloc[:, :4]
for c in ('stim_id', 'block', 'trial'):
    ann[c] = pd.to_numeric(ann[c], errors='coerce')

print(ann.head(12).to_string())
print('\nconditions:', ann['condition'].value_counts().to_dict())
print('blocks:', ann['block'].unique())
print('trial index: %s -> %s, monotonic=%s'
      % (ann['trial'].min(), ann['trial'].max(), ann['trial'].is_monotonic_increasing))

### Find the stimulus identity

EEGDash caches the BIDS tree, so the sidecar is on disk after the dataset loads. Locate it, read it,
and find the column that names which image was shown.

In [ ]:
import pandas as pd

ev_files = sorted(CACHE.rglob('*_events.tsv'))
print(f'{len(ev_files)} events.tsv found')
for f in ev_files[:3]:
    print('  ', f.relative_to(CACHE))

assert ev_files, 'no events.tsv in cache - check what EEGDash actually downloaded'
ev = pd.read_csv(ev_files[0], sep='\t')
print(f'\nshape: {ev.shape}')
print('columns:', list(ev.columns))
print(ev.head(10).to_string())

In [ ]:
# Which column identifies the image? Look for one with many distinct values.
for c in ev.columns:
    n = ev[c].nunique()
    print(f'{c:>24}  {n:>6} unique  e.g. {list(ev[c].dropna().unique()[:4])}')

# Also: what are the 61 annotations, if not stimuli?
print('\nannotation descriptions:', raw.annotations.description[:20].tolist())

In [ ]:
# Any stimulus->file mapping at the dataset root (THINGS-EEG2 used stimuli.tsv).
for pat in ('stimuli.tsv', 'participants.tsv', '*_events.json', 'dataset_description.json'):
    for f in sorted(CACHE.rglob(pat))[:2]:
        print('--', f.relative_to(CACHE))
        if f.suffix == '.tsv':
            print(pd.read_csv(f, sep='\t').head(5).to_string())
        else:
            print(json.dumps(json.load(open(f)), indent=2)[:600])

---

## 0.6 — Where is the image stream? (parked)

**Result: durations are unimodal.** 255-409 s, median 308, one peak. There is no separate class of
image runs, so per-image identity is not a matter of picking different recordings.

**The per-file event count test did not actually run.** `EEGDashDataset` is lazy: it fetches an EDF
when you touch `.raw`. Section 0.6 built the dataset but never accessed it, so the glob found only
the single file Stage 0 had already pulled. Cell below forces the download if we return to this.

**Parked, because Stage 1 does not depend on it.** NeuralBench ships a registered `xu2025alljoined`
config for the `eeg image` task and handles target extraction itself. Hand-rolling the stimulus
mapping is only needed if that config turns out to be broken or absent. Run Stage 1 first.

Open question recorded in `PLAN.md` §4b.

In [ ]:
# Forces the fetch that 0.6 assumed. Only needed if we return to hand-rolled extraction.
# for d in ds_probe.datasets:
#     _ = d.raw
# rows = [(f.name, len(pd.read_csv(f, sep='\t'))) for f in sorted(CACHE.rglob('*_events.tsv'))]
# print(pd.DataFrame(rows, columns=['file', 'n_events']).to_string())

In [ ]:
# Duration distribution across all records. Costs nothing: it is in the metadata already.
import numpy as np

durs = np.array([r.get('duration_seconds') or 0 for r in clean], dtype=float)
print(f'{len(durs)} records')
print('percentiles (s):', {q: round(float(np.percentile(durs, q)), 1)
                           for q in (0, 10, 50, 90, 100)})

hist, edges = np.histogram(durs, bins=12)
for h, lo, hi in zip(hist, edges[:-1], edges[1:]):
    print(f'{lo:7.0f}-{hi:7.0f}s  {h:5d}  {"#" * min(60, h // 5)}')

In [ ]:
# If durations are bimodal, the long mode is where the image trials live.
long_cut = float(np.percentile(durs, 75))
long_records = [r for r in clean if (r.get('duration_seconds') or 0) > long_cut]
print(f'{len(long_records)} records longer than {long_cut:.0f}s')

from collections import Counter
print('by session:', Counter(r['session'] for r in long_records))
print('by run    :', dict(sorted(Counter(r['run'] for r in long_records).items())[:12]))

### Sample across runs and count events in each

A handful of recordings from different runs and sessions. Whichever has ~1000 rows is the image
stream; if none do, identity is stored outside `events.tsv`.

In [ ]:
import pandas as pd

probe = []
seen_runs = set()
for r in clean:
    if r['subject'] != subj or r['run'] in seen_runs:
        continue
    seen_runs.add(r['run'])
    probe.append(r)
    if len(probe) == 6:
        break

for r in probe:
    print(f"ses-{r['session']} run-{r['run']}  {r.get('duration_seconds'):>7.1f}s  {r['bids_relpath']}")

In [ ]:
ds_probe = EEGDashDataset(records=probe, cache_dir=CACHE)

rows = []
for f in sorted(CACHE.rglob('*_events.tsv')):
    ev = pd.read_csv(f, sep='\t')
    tt = ev['trial_type'].astype(str) if 'trial_type' in ev else pd.Series(dtype=str)
    rows.append({
        'file': f.name,
        'n_events': len(ev),
        'conditions': tt.str.split(',').str[0].value_counts().to_dict(),
        'columns': list(ev.columns),
    })
print(pd.DataFrame(rows).to_string())

In [ ]:
# If some file has ~1000 rows, inspect it. That is the stimulus stream.
big = max(sorted(CACHE.rglob('*_events.tsv')), key=lambda f: len(pd.read_csv(f, sep='\t')))
ev = pd.read_csv(big, sep='\t')
print(big.name, ev.shape)
print(ev.head(15).to_string())
for c in ev.columns:
    print(f'{c:>12}  {ev[c].nunique():>6} unique')

---

# Stage 1 — NeuralBench, GPU, competition harness

Everything below needs a GPU and pulls the full Alljoined corpus. **Do not run until Stage 0 is
green** and the `# VERIFY` cells have been filled in from its output.

---
## [0] Environment check

Fail loudly and early. NeuralBench needs Python ≥ 3.12 and has **no CPU fallback** — every training run, `--debug` included, requires a working GPU.

In [ ]:
import sys, subprocess

print('Python:', sys.version)
assert sys.version_info >= (3, 12), (
    f'NeuralBench requires Python >=3.12, got {sys.version_info.major}.{sys.version_info.minor}. '
    'Fallback: uv venv --python 3.12'
)

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU. NeuralBench has no CPU fallback.'
print('torch:', torch.__version__, '| cuda:', torch.version.cuda)
# A driver/torch mismatch raises here rather than failing silently later.
print('capability:', torch.cuda.get_device_capability(0))
print('device:', torch.cuda.get_device_name(0))

---
## [1] Persistent paths

The `--prepare` cache is the expensive artifact and Colab runtimes are ephemeral. Put it on Drive
or rebuild it every session.

Track 1 needs room for: Alljoined-1.6M (~7.7 GB), the preprocessing cache, the DINOv2-giant
checkpoint (~4.5 GB), and one frozen embedding per unique stimulus.

In [ ]:
import os
from pathlib import Path

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/neurips26-eeg')
else:
    ROOT = Path('/content/neurips26-eeg')

DATA_DIR  = ROOT / 'data'
CACHE_DIR = ROOT / 'cache'
SAVE_DIR  = ROOT / 'results'
for d in (DATA_DIR, CACHE_DIR, SAVE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# HF checkpoints are large; keep them on Drive too so they survive a restart.
os.environ['HF_HOME'] = str(ROOT / 'hf')

import shutil
free_gb = shutil.disk_usage('/content').free / 1e9
print(f'ROOT={ROOT}\nfree on /content: {free_gb:.1f} GB')

---
## [2] Install

Pin versions so a green run stays green.

In [ ]:
%pip install -q neuralbench==0.3.1
# Track 1 target extraction pulls DINOv2 through transformers/torchvision.
%pip install -q 'transformers' 'torchvision' 'pillow'

import importlib.metadata as m
print('neuralbench:', m.version('neuralbench'))

In [ ]:
# VERIFY: confirm how neuralbench picks up storage locations in 0.3.1 —
# the docs describe configuring DATA_DIR / CACHE_DIR / SAVE_DIR at install time,
# and expose neuralbench.config_manager.setup_config / load_config / get_config.
os.environ['DATA_DIR']  = str(DATA_DIR)
os.environ['CACHE_DIR'] = str(CACHE_DIR)
os.environ['SAVE_DIR']  = str(SAVE_DIR)

!neuralbench --help

---
## [2b] Subsetting config

One knob. Works on any dataset at any size, so the same notebook runs as a 2% smoke test or a
full training run without edits.

Two independent levers, because download and load are different problems:

- **record selection** happens *before* download — the only way to touch a large corpus from a
  Colab disk. EEGDash queries exact recordings.
- **group subsetting** happens *after* download — a deterministic fraction of what is on disk,
  sampled over whole groups so split semantics survive.

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent / 'src'))  # repo src/ when run from notebooks/

from subset import (
    SubsetSpec, subset_groups, chance_top_k, describe,
    assert_disjoint, assert_gallery_intact, TRACK1_RULE,
)

FRACTION = 0.02          # 1.0 for a full run
SEED     = 33

spec = SubsetSpec(
    fraction=FRACTION,
    seed=SEED,
    group_key='subject',   # NOT image_id - see the gallery trap above
    notes=TRACK1_RULE,
)
print(spec)

---
## [3] Data — Alljoined-1.6M

Registered as `xu2025alljoined`. 20 participants, 130 h, 32 ch @ 256 Hz, 7.7 GB.

The size guard below exists because the *default* dataset for this task is THINGS-EEG2 at
**220 GB**. Omitting `--dataset` starts that download. Do not omit it.

In [ ]:
DATASET = 'xu2025alljoined'   # NOT the default. Default = Gifford2022Large @ 220 GB.
TASK    = 'eeg image'

EXPECTED_GB = 7.7
assert free_gb > EXPECTED_GB * 3, (
    f'Need roughly 3x {EXPECTED_GB} GB for raw + cache + checkpoint; only {free_gb:.1f} GB free.'
)
print(f'Downloading {DATASET} (~{EXPECTED_GB} GB). Safe to interrupt and re-run — it skips '
      'files already on disk.')

In [ ]:
!neuralbench eeg image --dataset {DATASET} --download

### [3b] Prepare the cache

**This is the only `--prepare` of the four tracks that needs a GPU** — it runs DINOv2-giant over
every unique stimulus to build the frozen target embeddings, alongside the usual window
preprocessing. Embeddings are content-keyed and shared across image tasks, so this cost is paid
once.

The docs quote SLURM-parallel timings (10 and 128 jobs). On Colab this runs serially and in-process,
so expect it to take proportionally longer.

In [ ]:
import time
t0 = time.time()
!neuralbench eeg image --dataset {DATASET} --prepare
print(f'prepare took {(time.time()-t0)/60:.1f} min')

---
## [4] Montage and input adaptation

Alljoined is 32-ch Emotiv @ 256 Hz. Encoder native rates differ — REVE is 200 Hz, LUNA is 256 Hz —
so log the resample path explicitly rather than letting a silent mismatch cost accuracy.

Also confirm channel positions actually resolve. When `ch_locs` are missing or NaN, the
position-derived encoding degrades to all-`INVALID_VALUE`, and topology-aware encoders quietly
lose the thing that makes them topology-aware.

In [ ]:
# VERIFY: adapt to however the prepared cache exposes montage info in 0.3.1.
# Intent: assert positions are real before trusting any position-aware encoder.

DATA_SFREQ = 256      # Alljoined-1.6M, matches the hidden Emotiv cohort
N_CHANS    = 32

ENCODER_NATIVE_SFREQ = {'eegnet': None, 'reve': 200, 'NtLuna': 256}

def log_resample_path(model_key):
    native = ENCODER_NATIVE_SFREQ.get(model_key)
    if native is None:
        print(f'{model_key}: no fixed native rate; runs at data rate {DATA_SFREQ} Hz')
    elif native == DATA_SFREQ:
        print(f'{model_key}: native {native} Hz == data {DATA_SFREQ} Hz — no resample')
    else:
        print(f'{model_key}: data {DATA_SFREQ} Hz -> encoder {native} Hz — RESAMPLE, verify it happens')

for k in ENCODER_NATIVE_SFREQ:
    log_resample_path(k)

---
## [5] Shape check, then debug run

`check_model` is a seconds-long shape probe. Run it before spending anything on training.

In [ ]:
from neuralbench import check_model

# VERIFY: signature per docs is check_model(model, modality, task).
# Using the stock baseline first — no gated weights, fastest path to a green pipeline.
# print(check_model(my_model, 'eeg', 'image'))

### Apply the subset, then guard the gallery

Run this before any training. If the gallery moved, the score is not comparable to anything and
the run should stop here.

In [ ]:
# VERIFY: pull the training-split group ids and the test gallery size from the prepared cache.
# train_groups = ...   # subject id per training window
# GALLERY_FULL = ...   # candidate count with no subsetting

# mask = subset_groups(train_groups, spec)
# print(describe(train_groups, mask, spec, gallery_size=GALLERY_FULL))
# assert_gallery_intact(gallery_size=GALLERY_FULL, expected=GALLERY_FULL)
# assert_disjoint(train_image_ids, test_image_ids)   # cross-stimulus split intact

In [ ]:
# 2 epochs, data subset, one seed, always in-process. ~2 min on a V100 with a warm cache.
!neuralbench eeg image --dataset {DATASET} -m eegnet --debug

---
## [6] Chance control — validates the metric independently of the dataset

This is the closest thing we get to metric parity on Track 1. Published chance for this task is
**Top-5 2.22 ± 0.31**. A random or shuffled predictor should land near that. If it lands far off,
the retrieval or scoring path is wrong and nothing downstream is trustworthy.

Note chance depends on gallery size, so treat this as an order-of-magnitude check on Alljoined
rather than an exact match to a THINGS-EEG2 number.

In [ ]:
# VERIFY: wire a shuffled/random-embedding predictor through the same scoring path
# used by the real run, so the metric code is what is being tested.
PUBLISHED_CHANCE_TOP5 = 2.22   # +/- 0.31, THINGS-EEG2
print(f'expect roughly {PUBLISHED_CHANCE_TOP5}% Top-5 from a random predictor')

---
## [7] Baseline — EEGNet

0.04 M params, ~2.5 h per seed on THINGS-EEG2 (expect less on the smaller Alljoined corpus).
This becomes **our** local reference on this corpus, since no published Alljoined number exists.

In [ ]:
t0 = time.time()
!neuralbench eeg image --dataset {DATASET} -m eegnet
print(f'eegnet took {(time.time()-t0)/60:.1f} min')

In [ ]:
# Turns cached metrics into comparison plots and CSV tables without retraining.
!neuralbench eeg image --dataset {DATASET} -m eegnet --plot-cached

---
## [8] The encoder: REVE

Selected on pretraining breadth, which is the property that should drive this choice.

| Encoder | Corpora | Scale | Channel handling |
|---|---|---|---|
| **REVE** | **92 datasets** | 60,000+ h, 25,000 subjects | 4D positional encoding over 3-D coords |
| LaBraM | 16 datasets | ~2,534 h | fixed 128-ch 10-20 order, unmatched channels dropped |
| BIOT | 6 datasets | - | Conv1d projection to 18 TCP bipolar channels |
| LUNA | TUEG + Siena | TUEG ~26,000 h | learned cross-attention unification |
| CBraMod | TUEG only | TUEG ~26,000 h | accepts any channel count |
| BENDR | TUEG only | TUEG ~26,000 h | fixed 20 channels |

REVE has 92 datasets against LaBraM's 16 and everyone else's 1-6. Three of the six are single-corpus
TUEG models, and TUEG is clinical pathology-screening EEG - far from healthy-subject viewing of
natural images, on top of being narrow.

**Montage handling is not a separate concern to defer.** You cannot pretrain across 92 heterogeneous
datasets without first solving montage invariance; they share no electrode layout. The narrow models
are TUEG-only precisely because one corpus with one montage is the easy case. REVE's positional
encoding is not a convenience bolted on - it is why the wide pretraining was possible.

The alternative has a concrete cost on this track: Alljoined is 32-channel consumer Emotiv. A
fixed-montage encoder either drops unmatched electrodes or interpolates onto a montage that was never
recorded. On 32 channels neither is affordable.

On leakage: REVE's image-task overlap is THINGS-EEG2, which taints its *published* 84.75. It never saw
Alljoined - what we train on, and where the hidden cohort comes from. The model is fine; the number is not.

**Second arm: LUNA.** Native 256 Hz matches Alljoined exactly where REVE resamples from 200 Hz, and it
reaches topology-invariance by a different mechanism. Useful as a check that results are not an artifact
of one encoder's inductive bias.

**Outside the zoo: ST-EEGFormer**, the KU Leuven model that won Challenge 1 in 2025 - roughly 13,300 h
across 11 datasets, open weights, proven in competition. Fewer corpora than REVE and it means leaving the
harness, so: back pocket, not starting point.

**Where the tuning effort probably belongs: the head and loss.** The target space is prescribed (frozen
DINOv2-giant, depth 0.6667, mean-pooled) and the stock config aligns to it with `ClipLoss`
(`norm_kind: y`, `temperature: false`, `symmetric: false`). Published EEG-to-image work (NICE, ATM,
NeuroCLIP) mostly varies the alignment head and objective rather than the backbone. `SigLipLoss` and
`DiffusionPrior` are already in the zoo.

Adaptation order: frozen probe as a floor, then LoRA. The 2025 winner found full fine-tuning overfit
while LoRA did not; EEG-FM-Compass found linear probing alone frequently insufficient. Run both ends.

In [ ]:
from huggingface_hub import login
login()   # needs a token with the REVE terms already accepted

In [ ]:
# Frozen probe first (configs/img/reve_frozen.yaml), then LoRA via the -w adaptation preset.
# VERIFY: exact -w preset names in 0.3.1.
!neuralbench eeg image --dataset {DATASET} -m reve
# !neuralbench eeg image --dataset {DATASET} -m reve -w lora
# !neuralbench eeg image --dataset {DATASET} -m NtLuna

---
## [9] Package the submission

Server-side evaluation is **inference-only** — the model arrives trained. A submission is a folder
holding `submission.py` plus weights, zipped.

For Track 1, `predict(X)` takes `(B, C, T)` and returns embeddings **`(B, 1536)`**.
Confirm against the Participation tab before the first real upload.

In [ ]:
SUBMISSION_DIR = ROOT / 'submissions' / 'track01_pipeline_proof'
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

# torch.save(model.state_dict(), SUBMISSION_DIR / 'weights.pt')

In [ ]:
solver = '''import torch

import benchmark_utils  # noqa: F401 - locates compet_core
from compet_core.base_solver import CompetSolver


class Solver(CompetSolver):
    name = "PipelineProof-T1"

    def load_model(self, meta):
        model = build_model(n_chans=meta["n_chans"], n_times=meta["n_times"],
                            n_outputs=meta["n_outputs"])  # 1536 for track 1
        state = torch.load(meta["weights_dir"] / "weights.pt",
                           map_location=meta["device"])
        model.load_state_dict(state)
        return model.to(meta["device"]).eval()
'''
(SUBMISSION_DIR / 'submission.py').write_text(solver)
print((SUBMISSION_DIR / 'submission.py').read_text())

### Local harness check

Once the starting kit is in hand, `Simulated` needs **zero download** — it runs anywhere,
including the MacBook. Cheapest possible check that our packaging is correct.

In [ ]:
# benchopt install tracks/image_decoding
# benchopt run tracks/image_decoding -d Simulated
# python codabench/ingestion_program/ingestion.py \
#     --submission-dir {SUBMISSION_DIR} --benchmark-dir tracks/image_decoding --datasets Simulated
# python codabench/scoring_program/scoring.py --prediction-dir output/ --output-dir scores/

---
## [10] Inference budget

Hard cap: a full test pass in **under 60 minutes on one H100/H200**. `duration` is a public
leaderboard column. The 2025 Challenge 1 winner lost Challenge 2 entirely to an inference timeout —
this is a real failure mode, not a formality.

In [ ]:
# VERIFY: time a full test pass through predict() on the largest available split.
BUDGET_MIN = 60
# elapsed = ...
# print(f'{elapsed:.1f} min / {BUDGET_MIN} min budget  ({elapsed/BUDGET_MIN:.0%})')

---
## [11] Run record

Rule 4 requires declaring every external corpus **and a compute estimate** in the final method
description. Adopting REVE means inheriting its 92-dataset pretraining corpus. Log it as you go —
reconstructing this in November is miserable.

In [ ]:
import json, datetime

record = {
    'track': '01-eeg-to-image',
    'date': datetime.date.today().isoformat(),
    'dataset': DATASET,
    'model': 'eegnet',
    'metric_key': 'test/full_retrieval/top5_acc_subject-agg',
    'score': None,
    'inference_min': None,
    'external_pretraining': [],   # e.g. REVE -> 92 public EEG datasets
    'compute_estimate_gpu_h': None,
    'notes': 'pipeline proof; no published Alljoined reference to match',
}
path = SAVE_DIR / f"run_{record['date']}_t01.json"
path.write_text(json.dumps(record, indent=2))
print(path)